> **Adaptación de CAIS, Intro to ML Safety.** Este notebook adapta al español el notebook L8 del curso *Intro to ML Safety* (course.mlsafety.org) del Center for AI Safety. Reproduced with permission of Center for AI Safety, Inc. CAIS no respalda este curso ni tiene relación con él.
>
> Las celdas marcadas «Basado en hendrycks/outlier-exposure» derivan de github.com/hendrycks/outlier-exposure (Dan Hendrycks y colaboradores, Apache License 2.0) y se han modificado (traducción y adaptación). La WideResNet (`wrn_with_pen.py`, hendrycks/pre-training, Apache-2.0) se descarga aparte. ViM (Wang et al., 2022) se implementa desde las ecuaciones del artículo.
>
> Avisos y licencias completos: `NOTICE.md` en la raíz del repositorio.

# Detección de anomalías

## Nota:
Se recomienda ejecutar el código con GPU. Si no hay una disponible, el código funciona igualmente en CPU, aunque puede tardar bastante más (en ese caso el código usa menos datos).

### Colaboradores del original:
Jason Ding

*Adaptación al español para el curso: los enunciados y los comentarios están traducidos y el código es el del original. Las celdas de Colab se han sustituido por una celda de requisitos para ejecución local.*

## Requisitos para ejecutar en local

El original corría en Google Colab: descargaba con `!wget` el modelo y una clase de red, e instalaba `torchvision==0.12.0` con `!pip`. Aquí se ejecuta en el equipo propio. Antes de empezar:

1. Un entorno de Python con Jupyter y estos paquetes: `numpy`, `scipy`, `scikit-learn`, `torch` y `torchvision` (y `gdown` para el modelo PixMix del final). Ejemplo: `pip install numpy scipy scikit-learn jupyter gdown`, y PyTorch con la orden que da su web para tu sistema y tu versión de CUDA (con una GPU NVIDIA va más rápido). El original fijaba `torchvision==0.12.0`, una versión que no sirve con Python 3.13: usa la versión que instale la orden de PyTorch.
2. Dos archivos en la misma carpeta que este notebook, que en el original bajaba `!wget`:
   - el modelo de CIFAR-10: <https://github.com/hendrycks/outlier-exposure/raw/master/CIFAR/snapshots/baseline/cifar10_wrn_baseline_epoch_99.pt>
   - la clase de la red: <https://raw.githubusercontent.com/hendrycks/pre-training/master/robustness/adversarial/models/wrn_with_pen.py>

   Descárgalos a mano (o con `curl -L -O <enlace>`) y, como `wrn_with_pen.py` es código que vas a importar, léelo antes de ejecutarlo.
3. Los conjuntos de datos (CIFAR-10, CIFAR-100 y SVHN) los descarga `torchvision` solo, en una carpeta `data`, la primera vez.
4. Este notebook **no se ha ejecutado** al preparar la adaptación. Puede haber fallos por versiones. Dos avisos: en PyTorch 2.6 o posterior `torch.load` usa `weights_only=True` por defecto y, si el archivo del modelo PixMix diera error, habría que pasar `weights_only=False` (solo si te fías del archivo); y en Windows, si los `DataLoader` con varios procesos se cuelgan, pon `prefetch = 0` en la celda de importaciones.

In [ ]:
# Requisitos para ejecución local (sustituye a las celdas de Colab con !wget y !pip).
import importlib.util
import os

archivos = ["cifar10_wrn_baseline_epoch_99.pt", "wrn_with_pen.py"]
faltan = [a for a in archivos if not os.path.exists(a)]
if faltan:
    raise FileNotFoundError(
        "Faltan estos archivos en la carpeta actual (" + os.getcwd() + "): " + ", ".join(faltan) + ". "
        "Descárgalos como explica la celda de texto anterior y vuelve a ejecutar esta celda."
    )
for modulo in ("torch", "torchvision", "numpy", "scipy", "sklearn"):
    if importlib.util.find_spec(modulo) is None:
        raise ModuleNotFoundError("Falta el paquete '" + modulo + "'. Instálalo como explica la celda de texto anterior.")
print("Requisitos básicos encontrados en:", os.getcwd())

## Prerrequisitos

En las celdas siguientes importamos las bibliotecas necesarias y cargamos el modelo de clasificación.

In [ ]:
import torch
import math
from torch.utils.data import Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torchvision import datasets
import numpy as np
import torch.nn.functional as F
import sklearn.metrics as sk
from wrn_with_pen import WideResNet

# Procesos de carga de datos (en Windows, si el DataLoader se cuelga, prueba con 0)
prefetch = 2

## Carga del modelo y de los datos

En las celdas siguientes cargamos el modelo de CIFAR-10. Además, cargamos CIFAR-10 y los conjuntos fuera de distribución.

In [ ]:
# Carga el modelo de CIFAR-10

net = WideResNet(depth=40, num_classes=10, widen_factor=2, dropRate=0.3)

if torch.cuda.is_available():
    net.load_state_dict(torch.load('cifar10_wrn_baseline_epoch_99.pt'))
    net.eval()
    net.cuda()
else:
    net.load_state_dict(torch.load('cifar10_wrn_baseline_epoch_99.pt', map_location=torch.device('cpu')))
    net.eval()

In [ ]:
# Basado en hendrycks/outlier-exposure (CIFAR/test.py), Apache License 2.0. Modificado.
# /////////////// Carga de los conjuntos de datos ///////////////
mean = [x / 255 for x in [125.3, 123.0, 113.9]]
std = [x / 255 for x in [63.0, 62.1, 66.7]]

# /////////////// CIFAR-10 ///////////////

data_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

cifar_10_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=data_transform
)

cifar10_data = cifar_10_data
cifar10_loader = torch.utils.data.DataLoader(cifar10_data, batch_size=200, shuffle=False,
                                          num_workers=prefetch, pin_memory=True)
ood_num_examples = len(cifar10_data) // 5

# /////////////// Ruido de Rademacher ///////////////

dummy_targets = torch.ones(ood_num_examples)
ood_data = torch.from_numpy(np.random.binomial(
    n=1, p=0.5, size=(ood_num_examples, 3, 32, 32)).astype(np.float32)) * 2 - 1
rademacher_ood_data = torch.utils.data.TensorDataset(ood_data, dummy_targets)
rademacher_ood_loader = torch.utils.data.DataLoader(rademacher_ood_data, batch_size=200, shuffle=True)

# /////////////// SVHN ///////////////

data_transform = transforms.Compose([transforms.Resize(32), transforms.ToTensor(), transforms.Normalize(mean, std)])

svhn_ood_data = torchvision.datasets.SVHN(root = "data", 
                          split="test",
                          transform = data_transform, 
                          download = True)

svhn_ood_loader = torch.utils.data.DataLoader(svhn_ood_data, batch_size=200, shuffle=True,
                                         num_workers=prefetch, pin_memory=True)

# /////////////// DTD ///////////////

# data_transform = transforms.Compose([transforms.Resize(32), transforms.CenterCrop(32),
#                                      transforms.ToTensor(), transforms.Normalize(mean, std)])

# dtd_ood_data = torchvision.datasets.DTD(root = "data", 
#                           split="test",
#                           transform = data_transform, 
#                           download = True)

# dtd_ood_loader = torch.utils.data.DataLoader(dtd_ood_data, batch_size=200, shuffle=True,
#                                          num_workers=prefetch, pin_memory=True)

# /////////////// CIFAR-100 ///////////////

data_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

cifar100_ood_data = datasets.CIFAR100(
    root="data",
    train=False,
    download=True,
    transform=data_transform
)

cifar100_ood_loader = torch.utils.data.DataLoader(cifar100_ood_data, batch_size=200, shuffle=True,
                                         num_workers=prefetch, pin_memory=True)

## Funciones generales

Las celdas siguientes definen funciones que usaremos en el resto del código. No tienes que hacer nada aquí.

In [ ]:
# Basado en hendrycks/outlier-exposure (CIFAR/test.py), Apache License 2.0. Modificado.
concat = lambda x: np.concatenate(x, axis=0)
to_np = lambda x: x.data.cpu().numpy()

'''
Calcula las puntuaciones de anomalía de una parte del conjunto de datos dado.
Si no hay GPU, se ejecuta sobre una fracción más pequeña del conjunto para
que los cálculos sean más rápidos.

loader: un DataLoader con los datos cargados de un conjunto
anomaly_score_calculator: función que recibe el logit de salida de un lote de
                          datos y/o la penúltima capa.
model_net: el modelo clasificador.
usePenultimate: True si anomaly_score_calculator necesita la penúltima capa
                como parámetro. False en caso contrario.
'''
def get_ood_scores(loader, anomaly_score_calculator, model_net, use_penultimate = False):
    _score = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(loader):
            if torch.cuda.is_available():
                fraction = 200
            else:
                fraction = 1000
            if batch_idx >= ood_num_examples // fraction:
                break

            if torch.cuda.is_available():
                data = data.cuda()

            output = model_net(data)

            if use_penultimate:
                score = anomaly_score_calculator(output[0], output[1])
            else:
                score = anomaly_score_calculator(output[0])
            _score.append(score)

    return concat(_score).copy()

In [ ]:
# Basado en hendrycks/outlier-exposure (CIFAR/test.py), Apache License 2.0. Modificado.
# /////////////// Impresión de resultados ///////////////
all_anomaly_results = {}

'''
Devuelve e imprime el AUROC de un conjunto de datos.

ood_loader: un DataLoader con los datos cargados de un conjunto
anomaly_score_calculator: función que recibe el logit de salida de un lote de
                          datos y/o la penúltima capa.
model_net: el modelo clasificador.
usePenultimate: True si anomaly_score_calculator necesita la penúltima capa
                como parámetro. False en caso contrario.
'''
def get_and_print_results(ood_loader, anomaly_score_calculator, model_net, use_penultimate):
    out_score = get_ood_scores(ood_loader, anomaly_score_calculator, model_net, use_penultimate = use_penultimate)
    auroc = get_auroc(out_score, in_score)
    print('AUROC: \t\t\t{:.2f}'.format(100 * auroc) + "%")
    return auroc

'''
Imprime el AUROC de todos los conjuntos fuera de distribución. Los resultados
se añaden a la variable global all_anomaly_results, que se usa más adelante
para mostrarlos.

anomaly_score_calculator: función que recibe el logit de salida de un lote de
                          datos y/o la penúltima capa.
anomaly_score_name: el nombre del método de puntuación de anomalía.
model_net: el modelo clasificador.
model_name: el nombre del modelo clasificador.
usePenultimate: True si anomaly_score_calculator necesita la penúltima capa
                como parámetro. False en caso contrario.
'''
def print_all_results(anomaly_score_calculator, anomaly_score_name, model_net, model_name = "default_model", use_penultimate = False):
    global in_score, all_anomaly_results
    in_score = get_ood_scores(cifar10_loader, anomaly_score_calculator, model_net, use_penultimate)
    results = []

    print('Rademacher Noise Detection')
    auroc = get_and_print_results(rademacher_ood_loader, anomaly_score_calculator, model_net, use_penultimate)
    results.append(auroc)

    print('\nSVHN Detection')
    auroc = get_and_print_results(svhn_ood_loader, anomaly_score_calculator, model_net, use_penultimate)
    results.append(auroc)

    # print('\nDTD Detection')
    # auroc = get_and_print_results(dtd_ood_loader, anomaly_score_calculator, model_net, use_penultimate)
    # results.append(auroc)

    print('\nCIFAR-100 Detection')
    auroc = get_and_print_results(cifar100_ood_loader, anomaly_score_calculator, model_net, use_penultimate)
    results.append(auroc)

    average = sum(results) / len(results)
    results.append(average)

    if not model_name in all_anomaly_results:
        all_anomaly_results[model_name] = {}
    all_anomaly_results[model_name][anomaly_score_name] = results

## Implementa el AUROC
Completa la función `get_auroc`. La usaremos para calcular el AUROC de un conjunto fuera de distribución.

Puede ayudarte la función `sklearn.metrics.roc_auc_score()`. Hay que usar tanto `_pos` como `_neg`.

In [ ]:
'''
Calcula el AUROC de un conjunto fuera de distribución.

_pos: array con las puntuaciones de anomalía del conjunto fuera de distribución
_neg: array con las puntuaciones de anomalía de las imágenes de CIFAR-10
return: el AUROC de los datos, en forma decimal
'''
def get_auroc(_pos, _neg):
    ############################################################################
    # TODO: calcula el AUROC.                                                  #
    ############################################################################

    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    return auroc_score

## Implementa las puntuaciones de anomalía
Completa las funciones siguientes, que calculan la puntuación de anomalía a partir de la salida del modelo para un lote de datos (la salida contiene un vector de logits por cada imagen del lote).

Las ecuaciones siguientes muestran cómo hay que transformar los logits para obtener la puntuación de anomalía.

Ecuación del logit máximo: <br>
$\text{Score}=-\text{max}\, l_k$

Ecuación del softmax máximo: <br>
$\text{Score}=-\text{max}\, p(y=k|x)$

Ecuación de la anomalía por entropía cruzada: <br>
$\text{Score} = \bar{l}-\text{log}\sum_{c=1}^{\text{num\_classes}}e^{l_c}$

(Aquí $\bar{l}$ es la media de los logits del ejemplo. Un aviso propio: esta puntuación es el opuesto de la entropía cruzada entre la distribución uniforme y la salida del modelo, y por eso es más alta cuanto más se parece la salida a un reparto uniforme.)

In [ ]:
# /////////////// Cálculo de las puntuaciones de anomalía ///////////////

'''
Calcula la puntuación de anomalía de logit máximo de un lote de salidas.

output: la salida del modelo para un lote de datos.
'''
def max_logit_anomaly_score(output):
    ############################################################################
    # TODO: calcula la puntuación de logit máximo (devuelve un array numpy).   #
    ############################################################################

    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    return

'''
Calcula la puntuación de anomalía de softmax máximo de un lote de salidas.

output: la salida del modelo para un lote de datos.
'''
def max_softmax_anomaly_score(output):
    ############################################################################
    # TODO: calcula la puntuación de softmax máximo (devuelve un array numpy). #
    ############################################################################

    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    return

'''
Calcula la puntuación de anomalía de entropía cruzada de un lote de salidas.

output: la salida del modelo para un lote de datos.
'''
def cross_entropy_anomaly_score(output):
    ############################################################################
    # TODO: calcula la puntuación de entropía cruzada (devuelve un array numpy).#
    ############################################################################

    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    return

## Muestra los resultados de AUROC

Ejecuta las celdas siguientes para ver qué tal funciona cada puntuación de anomalía con los conjuntos fuera de distribución.

In [ ]:
print("======= Max Logit AUROC Scores =======")
print_all_results(max_logit_anomaly_score, "Max Logit", net)

In [ ]:
print("======= Max Softmax AUROC Scores =======")
print_all_results(max_softmax_anomaly_score, "Max Softmax", net)

In [ ]:
print("======= Cross Entropy AUROC Scores =======")
print_all_results(cross_entropy_anomaly_score, "Cross Entropy", net)

## Implementa ViM

Vamos a implementar ViM (*virtual-logit matching*, coincidencia de logits virtuales). ViM funciona así.

Primero considera el **espacio principal** ($P$) de las características que se obtienen al pasar los datos de entrenamiento (CIFAR-10) por la red hasta justo antes de la última capa totalmente conectada. Esas características se centran en el origen de un nuevo sistema de coordenadas (definido por `u` en el código de abajo). Usaremos las 12 componentes principales más significativas de esas características como espacio principal.

ViM calcula **alfa** ($\alpha$) proyectando las características de los datos de entrenamiento sobre el espacio ortogonal al espacio principal ($P^\perp$). Esa proyección se llama **residuo** ($x^{P^\perp}$). PISTA: puede ayudarte `np.linalg.svd` para hallar las componentes principales (o puedes usar valores y vectores propios). El espacio ortogonal al principal se obtiene simplemente tomando las componentes restantes (todas menos las 12 primeras).

Alfa se calcula con esta ecuación:
> $\alpha := \frac{\sum^k_{i=1}\text{max}_{j=1,...,C}\{l^i_j\}}{\sum^K_{i=1}\| x_i^{P\perp} \|}$

Dicho con palabras: alfa es la suma de los logits máximos de cada dato, dividida entre la suma de las normas de sus residuos.

El alfa que calcules aquí debería salir en torno a 16,16.

In [ ]:
# Puede que te sirvan las funciones siguientes.
from numpy.linalg import pinv, norm
from scipy.special import logsumexp

_score = []
to_np = lambda x: x.data.cpu().numpy()
concat = lambda x: np.concatenate(x, axis=0)

# Extrae los pesos y los sesgos de la capa totalmente conectada
w, b = net.fc.weight.cpu().detach().numpy(), net.fc.bias.cpu().detach().numpy()
# Origen de un nuevo sistema de coordenadas del espacio de características, para quitar el sesgo
u = -np.matmul(pinv(w), b)

'''
Calcula y devuelve el espacio ortogonal al espacio principal
(principal_space_perp) y el valor de alfa, a partir de los datos de
entrenamiento del modelo.


training_data_loader: un DataLoader con los datos cargados de un conjunto de
                      entrenamiento.
model_net: el modelo clasificador.
verbose: si es True, imprime el valor de alfa.
return: devuelve el espacio ortogonal y el valor de alfa.
'''
def compute_ViM_principal_space_perp_and_alpha(training_data_loader, model_net, verbose = False):
    result = []

    # Toma el primer lote de los datos de entrenamiento para calcular principal_space_perp y alpha
    training_data, target = next(iter(training_data_loader))
    if torch.cuda.is_available():
        training_data = training_data.cuda()

    result = model_net(training_data)
    logit = result[0] # Logits (valores antes del softmax)
    penultimate = result[1] # Características / penúltima capa (valores antes de la capa totalmente conectada)

    logit_id_train = logit.cpu().detach().numpy().squeeze()
    feature_id_train = penultimate.cpu().detach().numpy().squeeze()

    ############################################################################
    # TODO: calcula el espacio ortogonal al espacio principal y después        #
    # calcula alfa.                                                            #
    ############################################################################
    if verbose:
        print('Computing principal_space_perp...')
    
    
    if verbose:
        print('Computing alpha...')


    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    if verbose:
        print(f'alpha = {alpha}')

    return principal_space_perp, alpha

principal_space_perp, alpha = compute_ViM_principal_space_perp_and_alpha(cifar10_loader, net, verbose = True)

## Implementa la puntuación de anomalía de ViM

Ahora implementa la puntuación de anomalía de ViM.

Primero, proyectamos las características (penúltima capa) sobre `principal_space_perp`, que hallamos antes: eso es el residuo. Multiplicamos la norma de ese residuo por alfa y obtenemos el **logit virtual**.

Después obtenemos lo que algunos llaman **puntuación de energía**, con el LogSumExp de los logits.

Por último, la **puntuación de anomalía** es el logit virtual menos la energía.

$\text{vlogit}= \alpha \| {x^{P^\perp}} \|$\
$\text{energy} = \text{ln}\sum_{i=1}^C e^{l_i}$\
$\text{anomaly\_score} = \text{vlogit} - \text{energy}$

In [ ]:
'''
Calcula la puntuación de anomalía de ViM de un lote de salidas.

output: la salida del modelo para un lote de datos.
penultimate: la penúltima capa (valores de las características) del modelo
             para un lote de datos.
'''
def ViM_anomaly_score_calculator(output, penultimate):
    logit_id_val = output.cpu().detach().numpy().squeeze()
    feature_id_val = penultimate.cpu().detach().numpy().squeeze()

    ############################################################################
    # TODO: calcula la puntuación de anomalía.                                 #
    ############################################################################

    ############################################################################
    #                             FIN DE TU CÓDIGO                             #
    ############################################################################
    
    return score_id

print("======= ViM_anomaly_score_calculator =======")
w, b = net.fc.weight.cpu().detach().numpy(), net.fc.bias.cpu().detach().numpy()
u = -np.matmul(pinv(w), b)
principal_space_perp, alpha = compute_ViM_principal_space_perp_and_alpha(cifar10_loader, net) # Comprueba que tienes bien los valores de ViM antes de calcular la puntuación 
print_all_results(ViM_anomaly_score_calculator, "ViM", net, use_penultimate = True)

## Compara los resultados de las puntuaciones de anomalía
Ejecuta la celda siguiente para ver cómo se comparan entre sí las puntuaciones en los conjuntos fuera de distribución. Según el original, ViM debería ser mejor que las demás en casi todos los conjuntos (el original dice "salvo CIFAR-10", pero los conjuntos fuera de distribución son ruido de Rademacher, SVHN y CIFAR-100; en la salida de referencia del final, ViM pierde con CIFAR-100).

In [ ]:
# Basado en hendrycks/outlier-exposure (CIFAR/test.py), Apache License 2.0. Modificado.
# ///////////////// Comparación de resultados /////////////////

def get_results_max(model_name = "normal"):
    all_anomaly_results[model_name]["max"] = [0,0,0,0,0]
    for key in all_anomaly_results[model_name].keys():
        if (key != "max"):
            index = 0
            for score in all_anomaly_results[model_name][key]:
                all_anomaly_results[model_name]["max"][index] = \
                    max(score, all_anomaly_results[model_name]["max"][index])
                index += 1


def compare_all_results():
    for model_name in all_anomaly_results:
        to_be_printed = " " * (25 - len(model_name)) + model_name
        dataset_names = ["Rademacher", "SVHN", "CIFAR-100", "Average"]
        for name in dataset_names:
            to_be_printed += " | " + " "*(6-math.ceil(len(name)/2)) + \
                                name + " "*(6-math.floor(len(name)/2))

        print(to_be_printed)
        print("=" * (25 + len(dataset_names) * 15))

        get_results_max(model_name = model_name)
        for key in all_anomaly_results[model_name].keys():
            if (key != "max"):
                to_be_printed = " "*(25-len(key)) + key
                index = 0
                for result in all_anomaly_results[model_name][key]:
                    if (all_anomaly_results[model_name]["max"][index] == result):
                        result = "*" + '{:.2f}'.format(round(result * 100, 2)) + "%"
                    else:
                        result = '{:.2f}'.format(round(result * 100, 2)) + "%"
                    to_be_printed += " | " + " "*(6-math.ceil(len(result)/2)) + \
                                        result + " "*(6-math.floor(len(result)/2))
                    index += 1
                print(to_be_printed)
        print()

    print("\n* highlights the maximum AUROC Score for an OOD Dataset")

compare_all_results()

## Aumento de datos

Ahora veremos si los modelos entrenados con métodos de aumento de datos para la robustez ayudan a detectar datos fuera de distribución.

Cargaremos un modelo de CIFAR-10 entrenado con el aumento de datos PixMix y veremos cómo se comporta frente al modelo por defecto, que no usó aumento de datos.

### Descarga del modelo PixMix (ejecución local)

En el original, dos celdas con `!gdown` bajaban de Google Drive el modelo PixMix y su clase de red. Aquí, con `gdown` instalado, se hace en una terminal, en la carpeta del notebook:

```
gdown 1mjIfbb3mfXXvAZ1sBnjotFr5yYFmLi68
gdown 1skZT6yplO-Sv4M8Ksgzx14cTY3H-HgOa
```

El código de abajo espera los nombres `checkpoint.pth.tar` (el modelo) y `wideresnet_with_pen.py` (la clase). Si Drive los baja con otro nombre, renómbralos. Como antes, lee el `.py` antes de importarlo.

In [ ]:
# Requisitos del modelo PixMix (sustituye a las celdas de Colab con !gdown).
import os

archivos = ["checkpoint.pth.tar", "wideresnet_with_pen.py"]
faltan = [a for a in archivos if not os.path.exists(a)]
if faltan:
    raise FileNotFoundError(
        "Faltan estos archivos en la carpeta actual (" + os.getcwd() + "): " + ", ".join(faltan) + ". "
        "Descárgalos con gdown como explica la celda de texto anterior."
    )
print("Archivos de PixMix encontrados.")

In [ ]:
from wideresnet_with_pen import WideResNet as WideResNet2

In [ ]:
# Carga el modelo PixMix
# Aviso de la adaptación: el original llama a net.eval() y net.cuda() (no a pixmix_net) en las
# ramas de abajo, así que pixmix_net queda en modo entrenamiento (dropout y normalización por lote
# activos), lo que puede alterar los resultados. Además, con GPU pixmix_net se queda en la CPU
# mientras los datos van a la GPU, y lo normal es que DataParallel dé un error de dispositivos
# (no probado). Se conserva el código tal cual; si quieres, añade pixmix_net.eval() (y, con GPU,
# pixmix_net.cuda()) tras cargar el checkpoint y compara.

pixmix_net = WideResNet2(depth=40, num_classes=10, widen_factor=4, drop_rate=0.3)
pixmix_net = torch.nn.DataParallel(pixmix_net)

if torch.cuda.is_available():
    checkpoint = torch.load('checkpoint.pth.tar')
    pixmix_net.load_state_dict(checkpoint['state_dict'])
    net.eval()
    net.cuda()
else:
    checkpoint = torch.load('checkpoint.pth.tar', map_location=torch.device('cpu'))
    pixmix_net.load_state_dict(checkpoint['state_dict'])
    net.eval()

## Prueba del modelo PixMix

Probemos ahora el modelo PixMix con las mismas puntuaciones de anomalía que programamos antes.

In [ ]:
print("======= Max Logit AUROC Scores =======")
print_all_results(max_logit_anomaly_score, "Max Logit", pixmix_net, model_name = "pixmix_trained_model")

In [ ]:
print("======= Max Softmax Probability AUROC Scores =======")
print_all_results(max_softmax_anomaly_score, "Max Softmax Probability", pixmix_net, model_name = "pixmix_trained_model")

In [ ]:
print("======= Cross Entropy AUROC Scores =======")
print_all_results(cross_entropy_anomaly_score, "Cross Entropy", pixmix_net, model_name = "pixmix_trained_model")

In [ ]:
print("======= ViM_anomaly_score_calculator =======")
w, b = pixmix_net.module.fc.weight.cpu().detach().numpy(), pixmix_net.module.fc.bias.cpu().detach().numpy()
u = -np.matmul(pinv(w), b)
principal_space_perp, alpha = compute_ViM_principal_space_perp_and_alpha(cifar10_loader, pixmix_net)
print_all_results(ViM_anomaly_score_calculator, "ViM", pixmix_net, model_name = "pixmix_trained_model", use_penultimate = True)

## Compara el modelo sin aumento de datos con el modelo PixMix

Veamos cómo se compara el modelo por defecto con el modelo PixMix ejecutando la celda siguiente.

Deberías ver que el modelo PixMix ayuda en la detección fuera de distribución y tiene un AUROC más alto (salvo con ViM). Esto sugiere que conviene evaluar los modelos con muchos conjuntos fuera de distribución para valorar bien su capacidad de detección.

In [ ]:
compare_all_results()

In [ ]:
# La salida de arriba debería parecerse a la de abajo (salida de referencia del notebook original,
# no es una ejecución de esta adaptación)

            default_model |  Rademacher  |     SVHN     |  CIFAR-100   |   Average   
                Max Logit |    69.99%    |    91.38%    |    87.18%    |    82.85%   
  Max Softmax Probability |    80.11%    |    92.50%    |   *88.14%    |    86.92%   
            Cross Entropy |    70.02%    |    90.85%    |    87.23%    |    82.70%   
                      ViM |   *99.97%    |   *95.70%    |    83.46%    |   *93.04%   

     pixmix_trained_model |  Rademacher  |     SVHN     |  CIFAR-100   |   Average   
                Max Logit |   *94.49%    |   *92.87%    |    86.63%    |   *91.33%   
  Max Softmax Probability |    92.16%    |    91.51%    |    87.28%    |    90.32%   
            Cross Entropy |    94.40%    |    92.39%    |    86.50%    |    91.10%   
                      ViM |    88.69%    |    92.17%    |   *90.39%    |    90.42%   


* highlights the maximum AUROC Score for an OOD Dataset


## Solución de referencia (añadido de la adaptación)

Las celdas siguientes no están en el original. Traen una posible solución de cada `TODO`, escrita para esta adaptación, por si te atascas. **Inténtalo primero por tu cuenta.** Están en texto (no se ejecutan) y **no se han ejecutado**: están escritas a partir de las ecuaciones del propio notebook, no se han probado con el modelo, y el valor de alfa (unos 16,16) no se ha comprobado. Copia cada bloque en la celda con su `TODO`.

### AUROC

Las anómalas (`_pos`) son la clase positiva y las puntuaciones altas deben corresponder a anómalas:

```python
def get_auroc(_pos, _neg):
    etiquetas = np.concatenate([np.ones(len(_pos)), np.zeros(len(_neg))])
    puntuaciones = np.concatenate([_pos, _neg])
    auroc_score = sk.roc_auc_score(etiquetas, puntuaciones)
    return auroc_score
```

### Las tres puntuaciones de anomalía

`output` llega como un tensor de PyTorch de forma (lote, clases). Las tres puntuaciones son más altas cuanto más anómalo parece el dato:

```python
def max_logit_anomaly_score(output):
    return -to_np(output.max(dim=1)[0])

def max_softmax_anomaly_score(output):
    return -np.max(to_np(F.softmax(output, dim=1)), axis=1)

def cross_entropy_anomaly_score(output):
    return to_np(output.mean(1) - torch.logsumexp(output, dim=1))
```

### ViM: espacio ortogonal y alfa

Se centran las características de entrenamiento en `u`, se hace la SVD (las filas de `vt` son las componentes principales, ordenadas de mayor a menor) y se quedan las componentes desde la 13 en adelante. Alfa es la media de los logits máximos dividida entre la media de las normas de los residuos, que es la razón de sumas de la ecuación:

```python
    if verbose:
        print('Computing principal_space_perp...')
    x = feature_id_train - u                              # características centradas en u
    _, _, vt = np.linalg.svd(x, full_matrices=True)       # filas de vt: componentes principales
    principal_space_perp = vt[12:].T                      # las componentes restantes: forma (D, D - 12)

    if verbose:
        print('Computing alpha...')
    residuo_train = norm(np.matmul(x, principal_space_perp), axis=-1)
    alpha = logit_id_train.max(axis=-1).mean() / residuo_train.mean()
```

### ViM: puntuación de anomalía

El logit virtual es alfa por la norma del residuo, y la energía es el logsumexp de los logits:

```python
    x = feature_id_val - u
    vlogit_id_val = norm(np.matmul(x, principal_space_perp), axis=-1) * alpha
    energy_id_val = logsumexp(logit_id_val, axis=-1)
    score_id = vlogit_id_val - energy_id_val
```

Para tus propias pruebas: una puntuación de anomalía correcta debería dar AUROC por encima de 0,5 en los tres conjuntos fuera de distribución; si te sale por debajo de 0,5, es probable que el signo esté cambiado.